In [10]:
from import_libs import *
from tqdm import tqdm

In [11]:
def entanglement(graph_state, total_qubits, avg_gates_per_layer, layers):

    random_ame_circuit = RandomAMECircuit(total_qubits=total_qubits, graph_state=graph_state,avg_gates_per_layer=avg_gates_per_layer)
    
    e = random_ame_circuit.entanglement_evolution(layers=layers)

    return e

def average_entanglement(graph_state, total_qubits, graph_qubit, avg_gates_per_layer, layers, num_shots):

    num_workers = -1
    entropy = Parallel(n_jobs=num_workers)(delayed(entanglement)(graph_state, total_qubits, avg_gates_per_layer, layers) for i in (range(num_shots)))
    
    S = np.zeros(layers, dtype = float)
    
    for i in range(num_shots):
        S += (1/num_shots)*np.array(entropy[i], dtype = float)

    return S


In [ ]:
graph_qubit = 4
total_qubits = 200
avg_gates_per_layer = int((total_qubits//graph_qubit)*0.5)
layers = 100
num_shots = 100


entropies = []  

for i in tqdm(range(len(Q4_graph))):
    graph_state = Q4_graph[i]
    # entropy = entanglement(graph_state, total_qubits, avg_gates_per_layer, layers)
    entropy  = average_entanglement(graph_state, total_qubits, graph_qubit, avg_gates_per_layer, layers, num_shots)
    entropies.append(entropy)
    

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
 
# Create cubehelix color palette
# palette = sns.cubehelix_palette(start=.5, rot=-.75, n_colors=len(entropies))
pal = sns.color_palette("colorblind", n_colors=len(entropies))

fig, ax = plt.subplots(figsize=(4,4))

t = np.arange(0,layers,1)

# Plot each entropy trace with cubehelix colors
for i in range(len(entropies)):
    j = i+20
    ax.plot(t, entropies[i], color=pal[i], label=rf'{i+1}')

# Formatting (keeping your original style)
# ax.loglog()
# ax.set_yscale('log')
# ax.set_xscale('log')

for spine in ax.spines.values():
    spine.set_edgecolor('gray')

plt.tick_params(top=True, left=True, right=True, bottom=True,
               direction="in", axis='both', which='both', 
               labelsize=15, color='k')

font2 = {'family':'serif', 'color':'black', 'size':18}
plt.xlabel(r"$t$", fontdict=font2)
plt.ylabel(r"$\langle S(t)\rangle $", fontdict=font2)
# Modified legend placement
plt.legend(
    fontsize=8,  # Slightly smaller font for external legend
    ncol=1,  # Fewer columns for vertical space
    loc='best', 
    # bbox_to_anchor=(1.02, 0.9),  # Positions legend outside axes
    frameon=False,
    borderaxespad=0.  # Removes padding between axes and legend
)
# ax.set_ylim(0.5,)
fig.set_dpi(120)
# plt.tight_layout(rect=[0, 0, 0.85, 1])  # Adjust right margin to make space
plt.show()

In [ ]:
# saturation_value = np.zeros(len(entropies), dtype = float)
# tolerance_value = 1
# saturation_time = np.zeros_like(saturation_value, dtype = int)

# for i in range(len(entropies)):
#     ent = entropies[i]
#     saturation_value[i] = ent[-1]
    
#     for j in range(len(ent)):
#         if np.abs(saturation_value[i] - ent[j]) <= tolerance_value:
#             saturation_time[i] = j 
#             break    



In [ ]:
def power_law(x, a):
    return a * (x)

power = []; growth_rate = []

i = 0
for ent in entropies:
    # fit_time = saturation_time[i]
    x = np.arange(1, layers+1, 1)
    y = ent
    params, covariance = curve_fit(power_law, x, y)
    a_fit = params
    growth_rate.append(a_fit)
    i += 1

In [ ]:
np.savetxt('ent_vel-g4.txt',growth_rate)

In [ ]:

# Pair each growth rate with its corresponding group label
groups_and_rates = list(zip([f'G{i+3}' for i in range(len(growth_rate))], Q4_graph, growth_rate))

# Sort by growth rate (ascending order)
sorted_groups = sorted(groups_and_rates, key=lambda x: x[2])

# Print sorted results
for group, gi, rate in sorted_groups:
    print(f'{group} {np.round(rate,4)}')
    # gi.draw_graph()

In [ ]:

# Pair each growth rate with its corresponding group label
groups_and_rates = list(zip([f'G{i+20}' for i in range(len(growth_rate))], Q7_graph, growth_rate))

# Sort by growth rate (ascending order)
sorted_groups = sorted(groups_and_rates, key=lambda x: x[2])

# Print sorted results
for group, gi, rate in sorted_groups:
    print(f'{group} {np.round(rate,4)}')
    # gi.draw_graph()

In [ ]:
i = 3
for Gi in Q4_graph:

    Gi.draw_graph(title = i)
    i +=1

In [12]:
def gf2_rank(mat):
    """Compute rank of a binary matrix over GF(2) using Gaussian elimination."""
    mat = mat.copy()
    rank = 0
    n_rows, n_cols = mat.shape
    
    for col in range(n_cols):
        pivot = -1
        # Find pivot row
        for row in range(rank, n_rows):
            if mat[row, col]:
                pivot = row
                break
        if pivot == -1:
            continue
            
        # Swap current row with pivot row
        mat[[rank, pivot]] = mat[[pivot, rank]]
        
        # Eliminate this column in other rows
        for row in range(n_rows):
            if row != rank and mat[row, col]:
                mat[row] = (mat[row] + mat[rank]) % 2
        rank += 1
    
    return rank


def binaryMatrix(zStabilizers):
    
    N = len(zStabilizers)
    Na = len(zStabilizers[0])
    binaryMatrix = np.zeros((N,2*Na))

    r = 0 # Row number
    
    for row in zStabilizers:
        c = 0 # Column number
        for i in row:
            if i == 3: # Pauli Z
                binaryMatrix[r,Na + c] = 1
            if i == 2: # Pauli Y
                binaryMatrix[r,Na + c] = 1
                binaryMatrix[r,c] = 1
            if i == 1: # Pauli X
                binaryMatrix[r,c] = 1
            c += 1
        r += 1

    return binaryMatrix


def entanglement(tab_forward, sysA , sysB, total_qubits) -> float:
        
    stabilizers = tab_forward.to_stabilizers()

    gA = [stim.PauliString([s[q] for q in sysA]) for s in stabilizers]
    # gB = [stim.PauliString([s[q] for q in sysB]) for s in stabilizers]

    na = len(sysA); nb = len(sysB)


    binary_matrix = binaryMatrix(gA)

    rank = gf2_rank(binary_matrix)
    
    return rank - na

In [14]:
n= 4
graph_qubit = [i for i in range(n)]

graph_ent = []

for Gi in Q4_graph:

    ent = 0
    for i in range(1,len(graph_qubit)):

        partA = graph_qubit[:i]
        partB = graph_qubit[i:]

        circuit = stim.Circuit()
        circuit.append('I', [n-1])

        Gi.apply_to_circuit(circuit, [i for i in range(4)], random_order=False)

        simulator = stim.TableauSimulator()
        simulator.do_circuit(circuit)
        tableau = simulator.current_inverse_tableau()
        tab_forward = tableau.inverse()

        ent += (1/(n-1))*entanglement(tab_forward, partA, partB, len(graph_qubit))

    
    graph_ent.append(ent)




# Pair each growth rate with its corresponding group label
groups_and_rates = list(zip([f'G{i+3}' for i in range(len(graph_ent))], Q4_graph, graph_ent))

# Sort by growth rate (ascending order)
sorted_groups = sorted(groups_and_rates, key=lambda x: x[2])

# Print sorted results
for group, gi, rate in sorted_groups:
    print(f'{group} {rate}')
    # gi.draw_graph()

G3 1.0
G4 1.0
